# Brownian Motion Analysis — From Text/CSV Positions (Batch Mode) v1.2

Batch-processes position files (Tracker TXT, MTrack2 CSV/TXT) with skip-if-up-to-date logic.
No OpenCV / no particle detection — positions are already tracked.

**Pipeline v1.2 changes:**
- MSD forced through origin (camera jitter handled via calibration slide)
- Sigma-clipping applied before MSD for consistency with variance method
- Dual-wall Fax\u00e9n correction (midplane + quarter-plane heights)
- Einstein/Batchelor viscosity correction for bead volume fraction
- Auto-discovery of `.txt` data files via `os.listdir`
- Modular cell structure with clear section headings
- Presentation PDFs saved to `presentation/` subfolder
- Filenames include slide prefix and trial number

In [ ]:
# ============================================================================
# CELL 1 — CONFIGURATION & JOBS (v1.2)
# ============================================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import chi2 as chi2_dist
from math import sqrt, pi
from pathlib import Path
from datetime import datetime
import configparser
import re
import os
import glob
import pandas as pd

# --- Pipeline version (bump to force reprocessing of all datasets) ---
PIPELINE_VERSION = '1.2'
FORCE_REPROCESS  = False
PRESENTATION_READY = True   # True = report-quality (plot_demo style)

# --- Universal constants ---
PIXEL_SIZE_DEFAULT   = 0.0684    # um/px (68.4 nm/px, 100x oil)
FPS_DEFAULT          = 29.0      # frames per second
TEMPERATURE_C_DEFAULT = 21.0     # Celsius
CHAMBER_DEPTH_UM     = 82.5      # tape spacer chamber depth (um)
MIN_SEGMENT_LENGTH   = 10        # min frames per usable segment
MAX_JUMP_UM          = 1.5       # max frame-to-frame jump (um) before splitting
k_B                  = 1.381e-23 # Boltzmann constant (J/K)
BEAD_DENSITY_KG_M3   = 1050.0    # polystyrene bead density

# --- Plot style (from PRESENTATION_READY flag) ---
if PRESENTATION_READY:
    _W_TEXT = 6.5
    _W1 = 0.80 * _W_TEXT;  _H1 = (2.0/3.0) * _W1
    _W2 = _W_TEXT;          _H2 = 3.5
    _W4 = _W_TEXT;          _H4 = _W_TEXT
    _FS_L = 12; _FS_LG = 10; _FS_T = 12
    _MS = 5;  _LW = 1.25; _CAP = 3; _SAVE_PDF = True
else:
    _W1 = 8;   _H1 = 6
    _W2 = 14;  _H2 = 6
    _W4 = 16;  _H4 = 12
    _FS_L = 12; _FS_LG = 9; _FS_T = 13
    _MS = 4;  _LW = 2; _CAP = 3; _SAVE_PDF = False

# --- Paths ---
ANALYSIS_ROOT = Path(os.path.abspath(os.path.dirname(__file__) if '__file__' in dir() else os.getcwd()))
DATA_ROOT = ANALYSIS_ROOT.parent / 'Data'
FIGURES_ROOT = ANALYSIS_ROOT / 'figures'

# --- Auto-discover .txt data files ---
_EXCLUDE_PATTERNS = ['noise', 'calibration', 'diffusion', 'onion']
_DATA_DIRS = sorted([d for d in DATA_ROOT.iterdir()
                     if d.is_dir() and re.match(r'^\d{4}-\d{2}-\d{2}$', d.name)])

JOBS = []
for _data_dir in _DATA_DIRS:
    for _txt_file in sorted(_data_dir.glob('*.txt')):
        _name_lower = _txt_file.name.lower()
        if any(ex in _name_lower for ex in _EXCLUDE_PATTERNS):
            continue
        # Must contain bead size marker (um or mu)
        if not re.search(r'\d+(?:_\d+)?\s*(?:mu|um)', _name_lower):
            continue
        JOBS.append((_txt_file, TEMPERATURE_C_DEFAULT))

print(f'Pipeline v{PIPELINE_VERSION} | Auto-discovered {len(JOBS)} data files')
print(f'Data root:    {DATA_ROOT}')
print(f'Figures root: {FIGURES_ROOT}')
for i, (f, t) in enumerate(JOBS):
    print(f'  [{i+1:2d}] {f.parent.name}/{f.name}')


## Configuration Notes

**`PRESENTATION_READY = True`**: Report-quality figures — 80% text-width (5.2"), 12pt labels, markersize 5, linewidth 1.25. PDFs saved to `presentation/` subfolder alongside PNGs.

**`FORCE_REPROCESS = False`**: Skip datasets whose `readme.txt` already has `pipeline_version = 1.2`. Set to `True` to reprocess everything.

**JOBS**: Auto-discovered from `Data/{date}/*.txt`. Excludes noise-calibration and non-bead files. Bead size, solute %, type, slide prefix, and trial number are all parsed from the filename automatically.

## Data Loading & Parsing Functions

In [ ]:
# ============================================================================
# DATA LOADING & PARSING FUNCTIONS
# ============================================================================

def parse_filename(filepath):
    """Parse experiment parameters from filename.

    Returns: (bead_um, solute_pct, solute_type, prefix, trial_num,
              stock_vol_ul, total_vol_ul)
    """
    name = Path(filepath).stem.lower()
    parts = name.split('-')

    # --- Prefix (slide ID): first token ---
    prefix = parts[0] if parts else ''

    # --- Trial number ---
    trial_match = re.search(r'trial(\d+)', name)
    trial_num = int(trial_match.group(1)) if trial_match else 1

    # --- Bead diameter ---
    bead_match = re.search(r'(\d+(?:_\d+)?)\s*(?:mu|um)', name)
    bead_um = float(bead_match.group(1).replace('_', '.')) if bead_match else None

    # --- Stock volume (bead suspension): number right after "0_5p-" ---
    stock_match = re.search(r'0_5p-(\d+(?:_\d+)?)ul', name)
    stock_vol_ul = float(stock_match.group(1).replace('_', '.')) if stock_match else None

    # --- Solute type + percentage + volumes ---
    solute_type = 'glycerol'
    solute_pct = 0.0
    water_vol = 0.0
    solute_vol = 0.0

    if re.search(r'[-_]ace[-_]|acetone', name):
        solute_type = 'acetone'
        ace_match = re.search(r'water-(\d+(?:_\d+)?)ul-ace(?:tone)?-(\d+(?:_\d+)?)(?:ul)?', name)
        if ace_match:
            water_vol = float(ace_match.group(1).replace('_', '.'))
            solute_vol = float(ace_match.group(2).replace('_', '.'))
            solute_pct = solute_vol / (water_vol + solute_vol) * 100
        else:
            ace2 = re.search(r'acetone-(\d+(?:_\d+)?)(?:ul)?', name)
            if ace2:
                solute_vol = float(ace2.group(1).replace('_', '.'))
                solute_pct = 100.0
    else:
        gly_match = re.search(r'water-(\d+(?:_\d+)?)ul-gly-(\d+(?:_\d+)?)(?:ul|l)?', name)
        if gly_match:
            water_vol = float(gly_match.group(1).replace('_', '.'))
            solute_vol = float(gly_match.group(2).replace('_', '.'))
            if solute_vol > 0:
                solute_pct = solute_vol / (water_vol + solute_vol) * 100
        elif re.search(r'gly-0(?:ul|l)?(?:\b|[-_]|$)', name):
            solute_pct = 0.0

    # --- Total volume ---
    total_vol_ul = None
    if stock_vol_ul is not None:
        total_vol_ul = stock_vol_ul + water_vol + solute_vol

    return bead_um, solute_pct, solute_type, prefix, trial_num, stock_vol_ul, total_vol_ul


def detect_format(filepath):
    """Auto-detect file format. Returns 'tracker' or 'mtrack2'."""
    with open(filepath, 'r') as f:
        first_line = f.readline().strip()
    return 'tracker' if first_line.startswith('#multi') else 'mtrack2'


def load_position_data(filepath, file_format):
    """Load position data from file.
    Returns: (data, n_frames, n_particles, fps_detected, input_units)
    """
    if file_format == 'tracker':
        return _load_tracker(filepath)
    else:
        return _load_mtrack2(filepath)


def _load_tracker(filepath):
    """Load Tracker format: #multi: header, t,x,y,x,y,... in micrometers."""
    with open(filepath, 'r') as f:
        lines = f.readlines()
    data_lines = lines[3:]
    rows, times = [], []
    for line in data_lines:
        line = line.strip()
        if not line:
            continue
        parts_l = line.split(',')
        times.append(float(parts_l[0]))
        vals = []
        for p in parts_l[1:]:
            p = p.strip()
            vals.append(float(p) if p else np.nan)
        rows.append(vals)
    fps = None
    if len(times) >= 2:
        dt_file = times[1] - times[0]
        if dt_file > 0:
            fps = round(1.0 / dt_file, 1)
    max_cols = max(len(r) for r in rows)
    for r in rows:
        r.extend([np.nan] * (max_cols - len(r)))
    data = np.array(rows)
    while data.shape[1] > 0 and np.all(np.isnan(data[:, -1])):
        data = data[:, :-1]
    if data.shape[1] % 2 != 0:
        data = data[:, :-1]
    return data, data.shape[0], data.shape[1] // 2, fps, 'um'


def _load_mtrack2(filepath):
    """Load MTrack2 format: Frame,X1,Y1,Flag1,... in pixels."""
    with open(filepath, 'r') as f:
        raw_lines = f.readlines()
    delimiter = '\t' if '\t' in raw_lines[0] else ','
    skip = 0
    for i in range(min(5, len(raw_lines))):
        stripped = raw_lines[i].strip()
        if not stripped:
            skip = i + 1; continue
        lower = stripped.lower()
        if 'frame' in lower or lower.startswith('track'):
            skip = i + 1
    data_lines = raw_lines[skip:]
    rows = []
    for line in data_lines:
        line = line.strip()
        if not line: continue
        parts_l = line.split(delimiter)
        remaining = parts_l[1:]
        xy = []
        for j in range(0, len(remaining), 3):
            x_str = remaining[j].strip() if j < len(remaining) else ''
            y_str = remaining[j+1].strip() if j+1 < len(remaining) else ''
            xy.append(float(x_str) if x_str else np.nan)
            xy.append(float(y_str) if y_str else np.nan)
        rows.append(xy)
    max_cols = max(len(r) for r in rows)
    for r in rows:
        r.extend([np.nan] * (max_cols - len(r)))
    data = np.array(rows)
    return data, data.shape[0], data.shape[1] // 2, None, 'px'


def split_tracks_at_jumps(data, min_length, max_jump):
    """Split multi-particle position data into clean segments."""
    segments = []
    n_particles = data.shape[1] // 2
    segment_id = 0
    for i in range(n_particles):
        x_raw = data[:, i * 2]
        y_raw = data[:, i * 2 + 1]
        valid_mask = ~np.isnan(x_raw) & ~np.isnan(y_raw)
        x_clean = x_raw[valid_mask]
        y_clean = y_raw[valid_mask]
        if len(x_clean) < min_length: continue
        dx = np.diff(x_clean); dy = np.diff(y_clean)
        steps = np.sqrt(dx**2 + dy**2)
        bad = np.where(steps > max_jump)[0]
        if len(bad) == 0:
            segment_id += 1
            segments.append({'x': x_clean, 'y': y_clean, 'id': segment_id,
                             'original_particle': i+1, 'length': len(x_clean)})
        else:
            x_segs = np.split(x_clean, bad + 1)
            y_segs = np.split(y_clean, bad + 1)
            for xs, ys in zip(x_segs, y_segs):
                if len(xs) >= min_length:
                    segment_id += 1
                    segments.append({'x': xs, 'y': ys, 'id': segment_id,
                                     'original_particle': i+1, 'length': len(xs)})
    segments.sort(key=lambda s: s['length'], reverse=True)
    return segments


def gaussian_pdf(x, mean, std_dev):
    """Normalized Gaussian PDF."""
    return (1.0 / (np.sqrt(2*np.pi) * abs(std_dev))) * np.exp(-(x-mean)**2 / (2*std_dev**2))

def sigma_clip(arr, sigma=3, max_iter=5):
    """Iterative sigma-clipping. Returns clipped array."""
    a = np.array(arr, dtype=float)
    for _ in range(max_iter):
        mu = np.mean(a); sd = np.std(a)
        if sd == 0: break
        mask = np.abs(a - mu) < sigma * sd
        if mask.all(): break
        a = a[mask]
    return a

def sigma_clip_mask(arr, sigma=3, max_iter=5):
    """Iterative sigma-clipping. Returns boolean mask (True = kept)."""
    a = np.array(arr, dtype=float)
    mask = np.ones(len(a), dtype=bool)
    for _ in range(max_iter):
        vals = a[mask]
        if len(vals) == 0: break
        mu = np.mean(vals); sd = np.std(vals)
        if sd == 0: break
        new_mask = np.abs(a - mu) < sigma * sd
        combined = mask & new_mask
        if np.array_equal(combined, mask): break
        mask = combined
    return mask

def linear_through_origin(t, slope):
    """MSD = slope * t  (forced through origin)."""
    return slope * t

def linear(t, slope, intercept):
    """MSD = slope * t + intercept  (kept for reference)."""
    return slope * t + intercept

def power_law(t, K, alpha):
    return K * t**alpha


print('Data functions loaded.')


## Viscosity & Theory Functions

In [ ]:
# ============================================================================
# VISCOSITY & THEORY FUNCTIONS
# ============================================================================

def get_glycerol_viscosity(glycerol_pct, temp_c):
    """Cheng (2008) formula for glycerol-water mixtures. Returns Pa.s."""
    T = temp_c
    eta_water = 1.790 * np.exp((-1230 - T) * T / (36100 + 360 * T))
    if glycerol_pct <= 0:
        return eta_water * 1e-3
    eta_glycerol = 12100.0 * np.exp((-1233 + T) * T / (9900 + 70 * T))
    cm = glycerol_pct / 100.0
    a = 0.705 - 0.0017 * T
    b = (4.9 + 0.036 * T) * a ** 2.5
    alpha_c = 1.0 - cm + (a * b * cm * (1 - cm)) / (a * cm + b * (1 - cm))
    eta_m = eta_water ** alpha_c * eta_glycerol ** (1 - alpha_c)
    return eta_m * 1e-3

def get_acetone_viscosity(acetone_vol_pct, temp_c):
    """Acetone-water mixture viscosity (Pa.s) via interpolation."""
    T = temp_c
    eta_water = 1.790 * np.exp((-1230 - T) * T / (36100 + 360 * T)) * 1e-3
    if acetone_vol_pct <= 0:
        return eta_water
    vol_pct = np.array([0, 5, 10, 15, 20, 30, 40, 50, 60, 80, 100])
    eta_mPa = np.array([0.978, 0.94, 0.90, 0.86, 0.82, 0.72, 0.62, 0.52, 0.44, 0.36, 0.32])
    return np.interp(acetone_vol_pct, vol_pct, eta_mPa) * 1e-3

def get_viscosity(solute_pct, temp_c, solute_type='glycerol'):
    if solute_type == 'acetone':
        return get_acetone_viscosity(solute_pct, temp_c)
    return get_glycerol_viscosity(solute_pct, temp_c)


def faxen_dual_wall(bead_radius_m, height_m, chamber_depth_m):
    """Dual-wall Faxen correction factor.
    F = 1 - (9/16)*s + (1/8)*s^3  where s = R/h + R/(L-h)
    Source: Diffusion Coefficient formulas.pdf (Nathan)
    """
    h = height_m
    L = chamber_depth_m
    s = bead_radius_m / h + bead_radius_m / (L - h)
    F = 1.0 - (9.0/16.0) * s + (1.0/8.0) * s**3
    return max(F, 0.01)  # safety floor


def einstein_viscosity_correction(eta_0_Pa_s, stock_vol_ul, total_vol_ul):
    """Einstein/Batchelor viscosity correction for bead volume fraction.
    eta_eff = eta_0 * (1 + 2.5*phi + 6.2*phi^2)
    phi = 0.005 * (V_stock / V_total) / 1.05
    Source: Batchelor (1977), formula from Diffusion Coefficient formulas.pdf
    Returns: (eta_corrected, phi)
    """
    if stock_vol_ul is None or total_vol_ul is None or total_vol_ul <= 0:
        return eta_0_Pa_s, 0.0
    phi = 0.005 * (stock_vol_ul / total_vol_ul) / 1.05
    eta_corrected = eta_0_Pa_s * (1 + 2.5 * phi + 6.2 * phi**2)
    return eta_corrected, phi


def compute_theory_D(bead_um, solute_pct, solute_type, temp_c,
                     stock_vol_ul=None, total_vol_ul=None,
                     chamber_depth_um=82.5):
    """Compute theoretical D with dual-wall Faxen + Einstein viscosity.

    Returns dict: D_0, D_mid, D_wall, F_mid, F_wall, eta, eta_corrected, phi
    All D values in um^2/s.

    Heights: midplane (L/2) and quarter-plane (L/8 = 10.3125 um per reference PDF).
    # NOTE: Gravitational settling height l_g = kT/(delta_rho * V_bead * g)
    # could refine the assumed particle height. Not implemented as it is not
    # in the reference PDF calculations. See Volpe & Volpe (2013).
    """
    T_K = temp_c + 273.15
    r_m = (bead_um / 2) * 1e-6
    L_m = chamber_depth_um * 1e-6

    # Base viscosity + Einstein correction
    eta_0 = get_viscosity(solute_pct, temp_c, solute_type)
    eta_corr, phi = einstein_viscosity_correction(eta_0, stock_vol_ul, total_vol_ul)

    # Stokes-Einstein (uncorrected)
    D_0 = k_B * T_K / (6 * pi * eta_corr * r_m) * 1e12  # um^2/s

    # Dual-wall Faxen at two heights
    h_mid  = L_m / 2          # midplane = 41.25 um
    h_wall = L_m / 8          # quarter-plane from PDF = 10.3125 um
    F_mid  = faxen_dual_wall(r_m, h_mid, L_m)
    F_wall = faxen_dual_wall(r_m, h_wall, L_m)

    D_mid  = D_0 * F_mid      # upper bound (less wall drag)
    D_wall = D_0 * F_wall     # lower bound (more wall drag)

    return {
        'D_0': D_0, 'D_mid': D_mid, 'D_wall': D_wall,
        'F_mid': F_mid, 'F_wall': F_wall,
        'eta': eta_0, 'eta_corrected': eta_corr, 'phi': phi,
    }


print('Physics functions loaded.')


## Displacement Analysis (Variance + Gaussian)

Computes D from frame-to-frame displacements using two methods:
1. **Direct Variance**: D = Var(dx)/(2dt)
2. **Gaussian Fit**: Fit PDF to histogram, D = sigma^2/(2dt)

In [ ]:
# ============================================================================
# DISPLACEMENT ANALYSIS (Variance + Gaussian)
# ============================================================================

def compute_displacement_D(segments, pixel_size, dt, sigma=3, max_iter=5):
    """Compute D from frame-to-frame displacements (variance + Gaussian fit).

    Returns dict with D_var, D_gauss, errors, histogram data, and
    per-segment clip masks for MSD consistency.
    """
    all_dx, all_dy = [], []
    for seg in segments:
        all_dx.extend(np.diff(seg['x']))
        all_dy.extend(np.diff(seg['y']))

    dx_um = np.array(all_dx) * pixel_size
    dy_um = np.array(all_dy) * pixel_size
    n_steps = len(dx_um)

    # --- Sigma-clip ---
    dx_clipped = sigma_clip(dx_um, sigma=sigma, max_iter=max_iter)
    dy_clipped = sigma_clip(dy_um, sigma=sigma, max_iter=max_iter)
    n_clipped = (n_steps - len(dx_clipped)) + (n_steps - len(dy_clipped))

    # --- METHOD 1: Direct Variance ---
    D_x_var = np.var(dx_clipped) / (2 * dt)
    D_y_var = np.var(dy_clipped) / (2 * dt)
    D_var = (D_x_var + D_y_var) / 2

    # --- METHOD 2: Gaussian Fit ---
    counts_x, edges_x = np.histogram(dx_clipped, bins='auto', density=True)
    centers_x = (edges_x[:-1] + edges_x[1:]) / 2
    try:
        popt_x, _ = curve_fit(gaussian_pdf, centers_x, counts_x, p0=[0, np.std(dx_clipped)])
        std_x_fit = abs(popt_x[1])
    except Exception:
        std_x_fit = np.std(dx_clipped)
        popt_x = [0, std_x_fit]

    counts_y, edges_y = np.histogram(dy_clipped, bins='auto', density=True)
    centers_y = (edges_y[:-1] + edges_y[1:]) / 2
    try:
        popt_y, _ = curve_fit(gaussian_pdf, centers_y, counts_y, p0=[0, np.std(dy_clipped)])
        std_y_fit = abs(popt_y[1])
    except Exception:
        std_y_fit = np.std(dy_clipped)
        popt_y = [0, std_y_fit]

    D_x_gauss = std_x_fit**2 / (2 * dt)
    D_y_gauss = std_y_fit**2 / (2 * dt)
    D_gauss = (D_x_gauss + D_y_gauss) / 2

    # --- Per-segment D for error bars ---
    seg_D_var, seg_D_gau = [], []
    for seg in segments:
        sdx = np.diff(seg['x']) * pixel_size
        sdy = np.diff(seg['y']) * pixel_size
        if len(sdx) < 5: continue
        sdx_c = sigma_clip(sdx, sigma=sigma, max_iter=3)
        sdy_c = sigma_clip(sdy, sigma=sigma, max_iter=3)
        seg_D = (np.var(sdx_c) + np.var(sdy_c)) / (4 * dt)
        seg_D_var.append(seg_D)
        seg_D_gau.append(seg_D)

    if len(seg_D_var) > 1:
        D_var_err = np.std(seg_D_var) / np.sqrt(len(seg_D_var))
        D_gauss_err = np.std(seg_D_gau) / np.sqrt(len(seg_D_gau))
    else:
        D_var_err = abs(D_var - D_gauss) / 2 if abs(D_var - D_gauss) > 0 else D_var * 0.1
        D_gauss_err = D_var_err

    return {
        'D_var': D_var, 'D_gauss': D_gauss,
        'D_var_err': D_var_err, 'D_gauss_err': D_gauss_err,
        'D_x_var': D_x_var, 'D_y_var': D_y_var,
        'D_x_gauss': D_x_gauss, 'D_y_gauss': D_y_gauss,
        'std_x_fit': std_x_fit, 'std_y_fit': std_y_fit,
        'popt_x': popt_x, 'popt_y': popt_y,
        'dx_clipped': dx_clipped, 'dy_clipped': dy_clipped,
        'counts_x': counts_x, 'edges_x': edges_x, 'centers_x': centers_x,
        'counts_y': counts_y, 'edges_y': edges_y, 'centers_y': centers_y,
        'n_steps': n_steps, 'n_clipped': n_clipped,
    }


print('Displacement analysis loaded.')


## MSD Analysis (Slope Method)

**Forced through origin**: `MSD = 4*D*tau` (no intercept). Camera jitter noise is handled separately via calibration slide measurement.

**Sigma-clipping** applied to trajectories before MSD computation to ensure consistency with the displacement (variance) method. This fixes the v1.1 bug where D_msd disagreed with D_var by 20-97%.

In [ ]:
# ============================================================================
# MSD ANALYSIS (with sigma-clipping + forced-through-origin fit)
# ============================================================================

def compute_msd(segments, pixel_size, dt, fps, sigma=3, max_iter=5,
                fit_fraction=0.25):
    """Compute MSD with sigma-clipping applied BEFORE MSD computation.

    BUG FIX (v1.2): Previous versions computed MSD on raw positions but
    variance on sigma-clipped displacements, causing systematic disagreement.
    Now: sigma-clip displacements -> build joint mask -> rebuild clean
    sub-segments -> compute MSD on clean data only.

    FIT: MSD = 4*D*tau (forced through origin). Camera jitter is handled
    separately via the calibration slide noise floor measurement.

    Returns dict with D_msd, alpha, MSD arrays, fit parameters.
    """
    max_lag = int(fps * 2.0)

    # --- Step 1: Sigma-clip displacements to build clean sub-segments ---
    # For each segment, identify outlier FRAMES using displacement clipping.
    # Joint mask: frame i is removed if EITHER dx[i] or dy[i] is an outlier.
    clean_segments = []
    for seg in segments:
        x = np.array(seg['x'], dtype=float)
        y = np.array(seg['y'], dtype=float)
        if len(x) < 5:
            continue
        dx = np.diff(x) * pixel_size
        dy = np.diff(y) * pixel_size
        # Get boolean masks (True = kept)
        mask_dx = sigma_clip_mask(dx, sigma=sigma, max_iter=max_iter)
        mask_dy = sigma_clip_mask(dy, sigma=sigma, max_iter=max_iter)
        # Joint: displacement i is OK only if BOTH dx[i] and dy[i] pass
        joint_disp_mask = mask_dx & mask_dy
        # Map displacement mask to position indices:
        # If displacement i is bad, positions i and i+1 are suspect.
        # We keep position j if ALL displacements touching it are OK.
        n_pos = len(x)
        pos_ok = np.ones(n_pos, dtype=bool)
        for j in range(len(joint_disp_mask)):
            if not joint_disp_mask[j]:
                pos_ok[j] = False
                pos_ok[j + 1] = False
        # Split remaining contiguous positions into clean sub-segments
        runs = []
        start = None
        for j in range(n_pos):
            if pos_ok[j]:
                if start is None:
                    start = j
            else:
                if start is not None:
                    if j - start >= MIN_SEGMENT_LENGTH:
                        runs.append((start, j))
                    start = None
        if start is not None and n_pos - start >= MIN_SEGMENT_LENGTH:
            runs.append((start, n_pos))
        for s, e in runs:
            clean_segments.append({'x': x[s:e], 'y': y[s:e],
                                    'length': e - s, 'id': seg['id']})

    if not clean_segments:
        # Fallback: use original segments if clipping removed everything
        clean_segments = [s for s in segments if s['length'] >= MIN_SEGMENT_LENGTH]

    # --- Step 2: Compute MSD on clean segments ---
    lag_times = np.arange(1, max_lag + 1) * dt
    all_MSDs = []
    for seg in clean_segments:
        x = np.array(seg['x'], dtype=float)
        y = np.array(seg['y'], dtype=float)
        MSD_seg = np.full(max_lag, np.nan)
        n_f = len(x)
        for lag in range(1, max_lag + 1):
            if n_f > lag:
                dx_lag = x[lag:] - x[:-lag]
                dy_lag = y[lag:] - y[:-lag]
                r_sq = (dx_lag * pixel_size)**2 + (dy_lag * pixel_size)**2
                MSD_seg[lag - 1] = np.mean(r_sq)
        all_MSDs.append(MSD_seg)

    all_MSDs = np.array(all_MSDs)
    # Use nanmean to handle segments shorter than max_lag
    with np.errstate(all='ignore'):
        MSD_um = np.nanmean(all_MSDs, axis=0)
        n_contributing = np.sum(~np.isnan(all_MSDs), axis=0)
        MSD_err = np.nanstd(all_MSDs, axis=0) / np.sqrt(np.maximum(n_contributing, 1))

    # Replace any remaining NaN with 0
    MSD_um = np.nan_to_num(MSD_um, nan=0.0)
    MSD_err = np.nan_to_num(MSD_err, nan=0.0)

    # --- Step 3: Linear fit forced through origin ---
    n_fit = max(3, max_lag // 4)
    fit_times = lag_times[:n_fit]
    fit_MSD = MSD_um[:n_fit]
    fit_err = np.where(MSD_err[:n_fit] > 0, MSD_err[:n_fit], 1e-10)

    try:
        popt_msd, pcov_msd = curve_fit(linear_through_origin, fit_times, fit_MSD,
                                        sigma=fit_err, absolute_sigma=True, p0=[1])
        slope = popt_msd[0]
        slope_err = np.sqrt(np.diag(pcov_msd))[0]
    except Exception:
        slope = np.sum(fit_MSD * fit_times) / np.sum(fit_times**2)
        slope_err = 0

    D_msd = slope / 4
    D_msd_err = slope_err / 4

    # --- Step 4: Power-law fit for alpha ---
    n_fit_power = max_lag // 2
    pl_t = lag_times[:n_fit_power]
    pl_msd = MSD_um[:n_fit_power]
    try:
        popt_pl, pcov_pl = curve_fit(power_law, pl_t, pl_msd,
                                      p0=[MSD_um[0]/lag_times[0] if lag_times[0]>0 else 1, 1.0])
        K_fit = popt_pl[0]; alpha_fit = popt_pl[1]
        alpha_err = np.sqrt(np.diag(pcov_pl))[1]
    except Exception:
        K_fit = MSD_um[0] / lag_times[0] if lag_times[0] > 0 else 1
        alpha_fit = 1.0; alpha_err = 0

    if alpha_fit < 0.9:
        motion_type = 'Sub-diffusive'
    elif alpha_fit > 1.1:
        motion_type = 'Super-diffusive'
    else:
        motion_type = 'Normal diffusion'

    return {
        'D_msd': D_msd, 'D_msd_err': D_msd_err,
        'alpha': alpha_fit, 'alpha_err': alpha_err,
        'motion_type': motion_type,
        'lag_times': lag_times, 'MSD_um': MSD_um, 'MSD_err': MSD_err,
        'n_fit': n_fit, 'n_fit_power': n_fit_power,
        'slope': slope, 'K_fit': K_fit,
        'clean_segments': clean_segments,
    }


print('MSD analysis loaded.')


## Plotting Functions

All plots use presentation style when `PRESENTATION_READY = True` (80% text-width, 12pt labels).

- PNGs saved to dataset folder (always)
- PDFs saved to `presentation/` subfolder (when `_SAVE_PDF = True`)
- Filenames include slide prefix + trial: e.g. `D_comparison-s2c-trial3.png`

In [ ]:
# ============================================================================
# PLOTTING FUNCTIONS (per-dataset)
# ============================================================================

def _save_figure(fig, save_dir, base_name, prefix, trial):
    """Save PNG (always) + PDF to presentation/ subfolder (if _SAVE_PDF)."""
    tag = f"-{prefix}-trial{trial}" if prefix else ""
    fig.savefig(str(save_dir / f"{base_name}{tag}.png"), dpi=200, bbox_inches='tight')
    if _SAVE_PDF:
        pres_dir = save_dir / 'presentation'
        pres_dir.mkdir(exist_ok=True)
        fig.savefig(str(pres_dir / f"{base_name}{tag}.pdf"), bbox_inches='tight')


def plot_trajectories(segments, pixel_size, pos_unit_label, px_label,
                      raw_data, n_particles, n_frames,
                      save_dir, prefix, trial):
    """Plot raw trajectories + displacement from start."""
    n_to_use = len(segments)
    colors = plt.cm.tab10(np.linspace(0, 1, max(n_to_use, 1)))
    fig, axes = plt.subplots(1, 2, figsize=(_W2, _H2))
    ax = axes[0]
    for seg, c in zip(segments, colors):
        ax.plot(seg['x'], seg['y'], '-', linewidth=_LW, color=c, alpha=0.7)
        ax.plot(seg['x'][0], seg['y'][0], 'o', color=c, markersize=_MS)
        ax.plot(seg['x'][-1], seg['y'][-1], 's', color=c, markersize=_MS)
    ax.set_xlabel(f'$X$ ({pos_unit_label})', fontsize=_FS_L)
    ax.set_ylabel(f'$Y$ ({pos_unit_label})', fontsize=_FS_L)
    ax.set_title(f'{n_to_use} Trajectories', fontsize=_FS_T)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax = axes[1]
    for seg, c in zip(segments, colors):
        ax.plot((seg['x'] - seg['x'][0]) * pixel_size,
                (seg['y'] - seg['y'][0]) * pixel_size,
                '-o', markersize=1, linewidth=1, color=c, alpha=0.7)
    ax.set_xlabel(r'$\Delta X$ ($\mu\mathrm{m}$)', fontsize=_FS_L)
    ax.set_ylabel(r'$\Delta Y$ ($\mu\mathrm{m}$)', fontsize=_FS_L)
    ax.set_title(f'Displacement from Start ({PIXEL_SIZE_DEFAULT*1000:.1f} nm/px)', fontsize=_FS_T)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    plt.tight_layout(); stamp_version(fig)
    _save_figure(fig, save_dir, 'trajectories', prefix, trial)
    plt.close(fig)


def plot_displacement_histograms(disp, dt, save_dir, prefix, trial):
    """Plot X and Y displacement histograms with Gaussian fits."""
    fig, axes = plt.subplots(1, 2, figsize=(_W2, _H2))
    bw_x = disp['edges_x'][1] - disp['edges_x'][0]
    ax = axes[0]
    ax.bar(disp['centers_x'], disp['counts_x'], width=bw_x, alpha=0.6,
           color='steelblue', label='Data (density)')
    x_fine = np.linspace(disp['centers_x'][0], disp['centers_x'][-1], 200)
    ax.plot(x_fine, gaussian_pdf(x_fine, *disp['popt_x']), 'r-', linewidth=_LW,
            label=f'Gaussian fit\n$\\sigma$ = {disp["std_x_fit"]:.4f}' + r' $\mu\mathrm{m}$')
    ax.set_xlabel(r'$\Delta x$ per frame ($\mu\mathrm{m}$)', fontsize=_FS_L)
    ax.set_ylabel(r'Probability Density', fontsize=_FS_L)
    ax.set_title(r'$X$ Displacement $-$ $D_x$ = ' + f'{disp["D_x_gauss"]:.4f}'
                 + r' $\mu\mathrm{m}^2/\mathrm{s}$', fontsize=_FS_T)
    ax.legend(fontsize=_FS_LG); ax.grid(True, alpha=0.3)

    bw_y = disp['edges_y'][1] - disp['edges_y'][0]
    ax = axes[1]
    ax.bar(disp['centers_y'], disp['counts_y'], width=bw_y, alpha=0.6,
           color='darkorange', label='Data (density)')
    y_fine = np.linspace(disp['centers_y'][0], disp['centers_y'][-1], 200)
    ax.plot(y_fine, gaussian_pdf(y_fine, *disp['popt_y']), 'r-', linewidth=_LW,
            label=f'Gaussian fit\n$\\sigma$ = {disp["std_y_fit"]:.4f}' + r' $\mu\mathrm{m}$')
    ax.set_xlabel(r'$\Delta y$ per frame ($\mu\mathrm{m}$)', fontsize=_FS_L)
    ax.set_ylabel(r'Probability Density', fontsize=_FS_L)
    ax.set_title(r'$Y$ Displacement $-$ $D_y$ = ' + f'{disp["D_y_gauss"]:.4f}'
                 + r' $\mu\mathrm{m}^2/\mathrm{s}$', fontsize=_FS_T)
    ax.legend(fontsize=_FS_LG); ax.grid(True, alpha=0.3)
    plt.tight_layout(); stamp_version(fig)
    _save_figure(fig, save_dir, 'displacement_histogram', prefix, trial)
    plt.close(fig)


def plot_msd_analysis(msd, dt, save_dir, prefix, trial):
    """Plot MSD linear + log-log with forced-through-origin fit."""
    lag_times = msd['lag_times']
    MSD_um = msd['MSD_um']; MSD_err = msd['MSD_err']
    n_fit = msd['n_fit']; n_fit_power = msd['n_fit_power']
    slope = msd['slope']; K_fit = msd['K_fit']; alpha_fit = msd['alpha']
    D_msd = msd['D_msd']; D_msd_err = msd['D_msd_err']
    alpha_err = msd['alpha_err']

    fit_line_t = np.linspace(0, lag_times[n_fit-1], 100)
    pl_line_t = np.linspace(dt, lag_times[n_fit_power-1], 100)
    valid = MSD_um > 0
    ref_t = np.logspace(np.log10(dt), np.log10(lag_times[-1]), 50)
    msd_at_1 = MSD_um[0] if MSD_um[0] > 0 else 1e-6

    fig, axes = plt.subplots(1, 2, figsize=(_W2, _H2))

    # Linear scale
    ax = axes[0]
    ax.errorbar(lag_times, MSD_um, yerr=MSD_err, fmt='o', markersize=_MS,
                capsize=_CAP, alpha=0.6, color='steelblue', label='MSD data', zorder=3)
    ax.plot(fit_line_t, linear_through_origin(fit_line_t, slope), 'r-', linewidth=_LW,
            label=r'Fit (origin): $D$ = ' + f'{D_msd:.4f}' + r' $\pm$ ' + f'{D_msd_err:.4f}'
            + r' $\mu\mathrm{m}^2/\mathrm{s}$')
    ax.plot(pl_line_t, power_law(pl_line_t, K_fit, alpha_fit), 'g--', linewidth=_LW,
            label=r'Power law: $\alpha$ = ' + f'{alpha_fit:.2f}')
    ax.set_xlabel(r'Lag time $\tau$ (s)', fontsize=_FS_L)
    ax.set_ylabel(r'MSD ($\mu\mathrm{m}^2$)', fontsize=_FS_L)
    ax.set_title(r'MSD $-$ Linear Scale (fit forced through origin)', fontsize=_FS_T)
    ax.legend(fontsize=_FS_LG); ax.grid(True, alpha=0.3)

    # Log-log scale
    ax = axes[1]
    ax.errorbar(lag_times[valid], MSD_um[valid], yerr=MSD_err[valid],
                fmt='o', markersize=_MS, capsize=_CAP, alpha=0.6, color='steelblue',
                label='MSD data', zorder=3)
    ax.plot(ref_t, msd_at_1 * (ref_t/dt)**1, 'k:', alpha=0.4, linewidth=_LW,
            label=r'$\alpha=1$ (diffusive)')
    ax.plot(ref_t, msd_at_1 * (ref_t/dt)**2, 'k--', alpha=0.4, linewidth=_LW,
            label=r'$\alpha=2$ (ballistic)')
    ax.plot(pl_line_t, power_law(pl_line_t, K_fit, alpha_fit), 'g-', linewidth=_LW,
            label=r'Fit: $\alpha$ = ' + f'{alpha_fit:.2f}' + r' $\pm$ ' + f'{alpha_err:.2f}')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'Lag time $\tau$ (s)', fontsize=_FS_L)
    ax.set_ylabel(r'MSD ($\mu\mathrm{m}^2$)', fontsize=_FS_L)
    ax.set_title(r'MSD $-$ Log-Log Scale', fontsize=_FS_T)
    ax.legend(fontsize=_FS_LG); ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout(); stamp_version(fig)
    _save_figure(fig, save_dir, 'msd_analysis', prefix, trial)
    plt.close(fig)


def plot_D_comparison(disp, msd, theory, bead_um, solute_pct, solute_label,
                      fps, save_dir, prefix, trial):
    """D comparison bar chart with theory range band [D_wall, D_mid]."""
    fig, ax = plt.subplots(figsize=(_W1, _H1))
    methods = ['Direct\nVariance', 'Gaussian\nFit', 'MSD\nSlope']
    D_vals = [disp['D_var'], disp['D_gauss'], msd['D_msd']]
    D_errs = [disp['D_var_err'], disp['D_gauss_err'], msd['D_msd_err']]
    bar_colors = ['steelblue', 'darkorange', 'forestgreen']
    x_pos = np.arange(len(methods))

    bars = ax.bar(x_pos, D_vals, yerr=D_errs, capsize=_CAP,
                  color=bar_colors, alpha=0.8, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, D_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(D_vals)*0.02,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

    # Theory range band [D_wall, D_mid]
    D_mid = theory['D_mid']; D_wall = theory['D_wall']
    ax.axhspan(D_wall, D_mid, alpha=0.2, color='crimson',
               label=f'Theory range: {D_wall:.4f}' + r'$-$' + f'{D_mid:.4f}'
               + r' $\mu\mathrm{m}^2/\mathrm{s}$')
    ax.axhline(D_mid, color='crimson', linestyle='-', linewidth=1.5, alpha=0.7)
    ax.axhline(D_wall, color='crimson', linestyle='--', linewidth=1.0, alpha=0.5)
    # Label lines
    ax.text(len(methods)-0.5, D_mid, f'  midplane ({D_mid:.4f})', va='bottom',
            fontsize=7, color='crimson')
    ax.text(len(methods)-0.5, D_wall, f'  quarter-plane ({D_wall:.4f})', va='top',
            fontsize=7, color='crimson')

    ax.set_xticks(x_pos); ax.set_xticklabels(methods)
    ax.set_ylabel(r'Diffusion Coefficient $D$ ($\mu\mathrm{m}^2/\mathrm{s}$)', fontsize=_FS_L)
    ax.set_title(f'{bead_um}' + r' $\mu\mathrm{m}$ beads $-$ '
                 + f'{solute_pct:.1f}' + f'% {solute_label} $-$ '
                 + f'{PIXEL_SIZE_DEFAULT*1000:.1f} nm/px $-$ {fps:.0f} fps', fontsize=_FS_T)
    ax.legend(fontsize=_FS_LG, loc='upper right')
    ax.grid(True, alpha=0.3, axis='y')
    all_D = D_vals + [D_mid]
    ax.set_ylim(0, max(all_D) * 1.35)
    plt.tight_layout(); stamp_version(fig)
    _save_figure(fig, save_dir, 'D_comparison', prefix, trial)
    plt.close(fig)


def plot_combined_summary(segments, raw_data, n_particles, n_frames,
                          disp, msd, theory, pixel_size, px_label, pos_unit_label,
                          bead_um, solute_label, solute_pct, temp_c, fps, file_format,
                          save_dir, prefix, trial):
    """2x3 combined summary grid."""
    n_to_use = len(segments)
    colors = plt.cm.tab10(np.linspace(0, 1, max(n_to_use, 1)))
    dt = 1.0 / fps
    suptitle = (f'{bead_um} um beads | {solute_pct:.1f}% {solute_label} | '
                f'{temp_c}C | {fps:.0f} fps | {file_format.upper()}')
    fig, axes = plt.subplots(2, 3, figsize=(12, 8) if PRESENTATION_READY else (24, 14))

    # [0,0] All raw positions
    ax = axes[0, 0]
    for i in range(n_particles):
        xr = raw_data[:, i*2]; yr = raw_data[:, i*2+1]
        v = ~np.isnan(xr) & ~np.isnan(yr)
        if np.sum(v) > 0: ax.plot(xr[v], yr[v], '.', markersize=1, alpha=0.3)
    ax.set_xlabel(f'X ({px_label})'); ax.set_ylabel(f'Y ({px_label})')
    ax.set_title(f'All positions ({n_particles} particles)'); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

    # [0,1] Segmented trajectories
    ax = axes[0, 1]
    for seg, c in zip(segments, colors):
        ax.plot(seg['x'], seg['y'], '-', linewidth=1, color=c, alpha=0.7)
        ax.plot(seg['x'][0], seg['y'][0], 'o', color=c, markersize=3)
    ax.set_xlabel(f'X ({px_label})'); ax.set_ylabel(f'Y ({px_label})')
    ax.set_title(f'{n_to_use} Trajectories'); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

    # [0,2] Displacement from start
    ax = axes[0, 2]
    for seg, c in zip(segments, colors):
        ax.plot((seg['x']-seg['x'][0])*pixel_size, (seg['y']-seg['y'][0])*pixel_size,
                '-o', markersize=1, linewidth=1, color=c, alpha=0.7)
    ax.set_xlabel(r'$\Delta X$ ($\mu$m)'); ax.set_ylabel(r'$\Delta Y$ ($\mu$m)')
    ax.set_title('Displacement from Start'); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

    # [1,0] X histogram
    ax = axes[1, 0]
    bw_x = disp['edges_x'][1] - disp['edges_x'][0]
    ax.bar(disp['centers_x'], disp['counts_x'], width=bw_x, alpha=0.6, color='steelblue')
    xf = np.linspace(disp['centers_x'][0], disp['centers_x'][-1], 200)
    ax.plot(xf, gaussian_pdf(xf, *disp['popt_x']), 'r-', linewidth=2,
            label=f'$\\sigma$={disp["std_x_fit"]:.4f} $\\mu$m')
    ax.set_xlabel(r'$\Delta x$ ($\mu$m)'); ax.set_ylabel('Density')
    ax.set_title('X Displacement Histogram'); ax.legend(fontsize=_FS_LG); ax.grid(True, alpha=0.3)

    # [1,1] MSD
    ax = axes[1, 1]
    lag = msd['lag_times']; MSD = msd['MSD_um']; MSD_e = msd['MSD_err']
    ax.errorbar(lag, MSD, yerr=MSD_e, fmt='o', markersize=3, capsize=2, alpha=0.6, color='steelblue')
    fit_t = np.linspace(0, lag[msd['n_fit']-1], 100)
    ax.plot(fit_t, linear_through_origin(fit_t, msd['slope']), 'r-', linewidth=2,
            label=f'D={msd["D_msd"]:.4f} $\\mu$m$^2$/s')
    pl_t = np.linspace(dt, lag[msd['n_fit_power']-1], 100)
    ax.plot(pl_t, power_law(pl_t, msd['K_fit'], msd['alpha']), 'g--', linewidth=2,
            label=f'$\\alpha$={msd["alpha"]:.2f}')
    ax.set_xlabel(r'$\tau$ (s)'); ax.set_ylabel(r'MSD ($\mu$m$^2$)')
    ax.set_title('MSD (forced through origin)'); ax.legend(fontsize=_FS_LG); ax.grid(True, alpha=0.3)

    # [1,2] D comparison
    ax = axes[1, 2]
    methods_s = ['D_var', 'D_gauss', 'D_msd']
    D_v = [disp['D_var'], disp['D_gauss'], msd['D_msd']]
    D_e = [disp['D_var_err'], disp['D_gauss_err'], msd['D_msd_err']]
    bars = ax.bar(methods_s, D_v, yerr=D_e, capsize=4,
                  color=['steelblue','darkorange','forestgreen'], alpha=0.8, edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, D_v):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(D_v)*0.02,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)
    ax.axhspan(theory['D_wall'], theory['D_mid'], alpha=0.2, color='crimson', label='Theory range')
    ax.set_ylabel(r'$D$ ($\mu$m$^2$/s)'); ax.set_title('D Comparison')
    ax.grid(True, alpha=0.3, axis='y'); ax.set_ylim(0, max(D_v + [theory['D_mid']]) * 1.35)
    ax.legend(fontsize=7)

    fig.suptitle(suptitle, fontsize=_FS_T + 2, fontweight='bold')
    plt.tight_layout(); stamp_version(fig)
    _save_figure(fig, save_dir, 'combined_summary', prefix, trial)
    plt.close(fig)


print('Plot functions loaded.')


## Trend Plot Functions

Lab-level plots show all 3 methods (Variance, Gauss, MSD) separately.
Per-date trends use grouped bar charts with clear method labeling.

In [ ]:
# ============================================================================
# TREND PLOT FUNCTIONS (overall, per-date, lab-level)
# ============================================================================

method_colors = {'D_var': 'steelblue', 'D_gauss': 'darkorange',
                 'D_msd': 'forestgreen', 'Theory': 'crimson'}

def plot_overall_trends(all_results, save_dir):
    """2x2 overall trend grid with clear method labeling."""
    if len(all_results) < 2:
        print('Need >= 2 datasets for overall trends.'); return
    fig, axes = plt.subplots(2, 2, figsize=(_W4, _H4))
    fig.suptitle(f'Overall Trends $-$ Pipeline v{PIPELINE_VERSION}', fontsize=_FS_T+2, weight='bold')

    # (0,0) Parity: D_exp vs D_theory — separate markers per method
    ax = axes[0, 0]
    d_the = [r['d_theory'] for r in all_results]
    ax.scatter(d_the, [r['d_var'] for r in all_results], c=method_colors['D_var'],
               s=40, marker='o', label='D (Variance)', zorder=3, alpha=0.7)
    ax.scatter(d_the, [r['d_gau'] for r in all_results], c=method_colors['D_gauss'],
               s=40, marker='s', label='D (Gaussian)', zorder=3, alpha=0.7)
    ax.scatter(d_the, [r['d_msd'] for r in all_results], c=method_colors['D_msd'],
               s=40, marker='^', label='D (MSD)', zorder=3, alpha=0.7)
    lim = max(max(d_the), max(r['d_var'] for r in all_results),
              max(r['d_msd'] for r in all_results)) * 1.15
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, label='1:1')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel(r'$D_{\mathrm{theory}}$ [$\mu\mathrm{m}^2/\mathrm{s}$]', fontsize=_FS_L)
    ax.set_ylabel(r'$D_{\mathrm{exp}}$ [$\mu\mathrm{m}^2/\mathrm{s}$]', fontsize=_FS_L)
    ax.set_title('Parity Plot (by method)', fontsize=_FS_T)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # (0,1) D vs viscosity — separate series per method
    ax = axes[0, 1]
    etas = [r['eta'] for r in all_results]
    ax.scatter(etas, [r['d_var'] for r in all_results], c=method_colors['D_var'],
               s=40, marker='o', label='D (Variance)', zorder=3, alpha=0.7)
    ax.scatter(etas, [r['d_gau'] for r in all_results], c=method_colors['D_gauss'],
               s=40, marker='s', label='D (Gaussian)', zorder=3, alpha=0.7)
    ax.scatter(etas, [r['d_msd'] for r in all_results], c=method_colors['D_msd'],
               s=40, marker='^', label='D (MSD)', zorder=3, alpha=0.7)
    ax.scatter(etas, [r['d_theory'] for r in all_results], c=method_colors['Theory'],
               s=60, marker='x', linewidths=2, label='Theory (midplane)', zorder=4)
    ax.set_xlabel(r'$\eta$ [$\mathrm{mPa{\cdot}s}$]', fontsize=_FS_L)
    ax.set_ylabel(r'$D$ [$\mu\mathrm{m}^2/\mathrm{s}$]', fontsize=_FS_L)
    ax.set_title(r'$D$ vs Viscosity (by method)', fontsize=_FS_T)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # (1,0) D vs bead size — separate series per method
    ax = axes[1, 0]
    beads = [r['bead'] for r in all_results]
    ax.scatter(beads, [r['d_var'] for r in all_results], c=method_colors['D_var'],
               s=40, marker='o', label='D (Variance)', zorder=3, alpha=0.7)
    ax.scatter(beads, [r['d_gau'] for r in all_results], c=method_colors['D_gauss'],
               s=40, marker='s', label='D (Gaussian)', zorder=3, alpha=0.7)
    ax.scatter(beads, [r['d_msd'] for r in all_results], c=method_colors['D_msd'],
               s=40, marker='^', label='D (MSD)', zorder=3, alpha=0.7)
    ax.scatter(beads, [r['d_theory'] for r in all_results], c=method_colors['Theory'],
               s=60, marker='x', linewidths=2, label='Theory', zorder=4)
    ax.set_xlabel(r'Bead diameter [$\mu\mathrm{m}$]', fontsize=_FS_L)
    ax.set_ylabel(r'$D$ [$\mu\mathrm{m}^2/\mathrm{s}$]', fontsize=_FS_L)
    ax.set_title(r'$D$ vs Bead Size (by method)', fontsize=_FS_T)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # (1,1) Alpha bar chart with error bars
    ax = axes[1, 1]
    labels = [f"{r.get('prefix','')}-t{r.get('trial',1)}" for r in all_results]
    alphas = [r['alpha'] for r in all_results]
    alpha_errs = [r.get('alpha_err', 0) for r in all_results]
    y_pos = np.arange(len(labels))
    ax.barh(y_pos, alphas, xerr=alpha_errs, color='mediumpurple',
            edgecolor='none', height=0.7, capsize=2)
    ax.set_yticks(y_pos); ax.set_yticklabels(labels, fontsize=6)
    ax.axvline(x=1.0, color='red', linestyle='--', alpha=0.7, label=r'$\alpha=1$')
    ax.set_xlabel(r'MSD Exponent $\alpha$', fontsize=_FS_L)
    ax.set_title(r'MSD Exponent $\alpha$', fontsize=_FS_T)
    ax.legend(fontsize=7)

    plt.tight_layout(); stamp_version(fig)
    fig.savefig(str(save_dir / 'overall_trends.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: overall_trends.png')


def plot_date_trends(date_results, date_str, save_dir):
    """Per-date grouped bar chart showing all 3 methods + theory band."""
    if len(date_results) < 2:
        return
    fig, ax = plt.subplots(figsize=(_W2, _H2))
    n = len(date_results)
    labels = [f"{r.get('prefix','?')}-t{r.get('trial',1)}" for r in date_results]
    x = np.arange(n)
    w = 0.2

    ax.bar(x - w, [r['d_var'] for r in date_results], w, color=method_colors['D_var'],
           label='Variance', alpha=0.8, edgecolor='black', linewidth=0.3)
    ax.bar(x,     [r['d_gau'] for r in date_results], w, color=method_colors['D_gauss'],
           label='Gaussian', alpha=0.8, edgecolor='black', linewidth=0.3)
    ax.bar(x + w, [r['d_msd'] for r in date_results], w, color=method_colors['D_msd'],
           label='MSD', alpha=0.8, edgecolor='black', linewidth=0.3)

    # Theory range bars
    for i, r in enumerate(date_results):
        d_mid = r.get('d_theory', 0)
        d_wall = r.get('d_wall', d_mid * 0.8)
        ax.plot([i - 1.5*w, i + 1.5*w], [d_mid, d_mid], '-', color='crimson', linewidth=1.5)
        ax.fill_between([i - 1.5*w, i + 1.5*w], d_wall, d_mid, alpha=0.15, color='crimson')

    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
    ax.set_ylabel(r'$D$ [$\mu\mathrm{m}^2/\mathrm{s}$]', fontsize=_FS_L)
    ax.set_title(f'Session {date_str} $-$ D by Method (red band = theory range)', fontsize=_FS_T)
    ax.legend(fontsize=_FS_LG); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout(); stamp_version(fig)
    date_dir = save_dir / date_str
    date_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(date_dir / f'trends_{date_str}.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: trends_{date_str}.png')


def plot_lab_level_trends(all_results, save_dir):
    """Lab-level plots: D vs viscosity by bead, D vs bead by viscosity."""
    bead_sizes = sorted(set(r['bead'] for r in all_results))
    glycerol_results = [r for r in all_results if r['solute_type'] == 'glycerol']
    if len(glycerol_results) < 2:
        print('Need >= 2 glycerol datasets for lab-level plots.'); return

    # D vs viscosity by bead size
    n_panels = len(bead_sizes)
    fig, axes_eta = plt.subplots(1, n_panels,
        figsize=(_W2, _H2) if PRESENTATION_READY else (6*n_panels, 5))
    if n_panels == 1: axes_eta = [axes_eta]
    fig.suptitle(r'$D$ vs Viscosity at Constant Bead Size (glycerol only)',
                 fontsize=_FS_T+2, weight='bold')
    for ax, bead in zip(axes_eta, bead_sizes):
        pts = [r for r in glycerol_results if r['bead'] == bead]
        if not pts: ax.set_visible(False); continue
        etas = [r['eta'] for r in pts]
        ax.scatter(etas, [r['d_var'] for r in pts], c=method_colors['D_var'], s=40,
                   label='D_var', zorder=3, alpha=0.7)
        ax.scatter(etas, [r['d_gau'] for r in pts], c=method_colors['D_gauss'], s=40,
                   label='D_gauss', marker='s', zorder=3, alpha=0.7)
        ax.scatter(etas, [r['d_msd'] for r in pts], c=method_colors['D_msd'], s=40,
                   label='D_msd', marker='^', zorder=3, alpha=0.7)
        ax.scatter(etas, [r['d_theory'] for r in pts], c=method_colors['Theory'], s=60,
                   label='Theory', marker='x', linewidths=2, zorder=4)
        _eta_r = np.linspace(min(etas)*0.8, max(etas)*1.2, 50)
        _r_m = (bead / 2) * 1e-6
        _D_curve = k_B * 293.15 / (6 * np.pi * (_eta_r * 1e-3) * _r_m) * 1e12
        ax.plot(_eta_r, _D_curve, 'r--', alpha=0.3, linewidth=1.5)
        ax.set_xlabel(r'$\eta$ [mPa$\cdot$s]', fontsize=_FS_L)
        ax.set_ylabel(r'$D$ [$\mu$m$^2$/s]', fontsize=_FS_L)
        ax.set_title(f'{bead}' + r' $\mu$m beads', fontsize=_FS_T)
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    plt.tight_layout(); stamp_version(fig)
    fig.savefig(str(save_dir / 'D_vs_viscosity_by_bead.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: D_vs_viscosity_by_bead.png')

    # D vs bead size by viscosity
    viscosities = sorted(set(r['solute_label'] for r in glycerol_results))
    n_v = len(viscosities)
    fig, axes_bead = plt.subplots(1, n_v,
        figsize=(_W2, _H2) if PRESENTATION_READY else (6*n_v, 5))
    if n_v == 1: axes_bead = [axes_bead]
    fig.suptitle(r'$D$ vs Bead Size at Constant Viscosity (glycerol only)',
                 fontsize=_FS_T+2, weight='bold')
    for ax, visc_label in zip(axes_bead, viscosities):
        pts = [r for r in glycerol_results if r['solute_label'] == visc_label]
        if not pts: ax.set_visible(False); continue
        beads_p = [r['bead'] for r in pts]
        ax.scatter(beads_p, [r['d_var'] for r in pts], c=method_colors['D_var'], s=40,
                   label='D_var', zorder=3, alpha=0.7)
        ax.scatter(beads_p, [r['d_gau'] for r in pts], c=method_colors['D_gauss'], s=40,
                   label='D_gauss', marker='s', zorder=3, alpha=0.7)
        ax.scatter(beads_p, [r['d_msd'] for r in pts], c=method_colors['D_msd'], s=40,
                   label='D_msd', marker='^', zorder=3, alpha=0.7)
        ax.scatter(beads_p, [r['d_theory'] for r in pts], c=method_colors['Theory'], s=60,
                   label='Theory', marker='x', linewidths=2, zorder=4)
        _d_r = np.linspace(0.5, max(beads_p)*1.2, 50)
        _eta_avg = np.mean([r['eta'] for r in pts]) * 1e-3
        _D_curve_b = k_B * 293.15 / (6 * np.pi * _eta_avg * (_d_r / 2 * 1e-6)) * 1e12
        ax.plot(_d_r, _D_curve_b, 'r--', alpha=0.3, linewidth=1.5)
        ax.set_xlabel(r'Bead diameter [$\mu$m]', fontsize=_FS_L)
        ax.set_ylabel(r'$D$ [$\mu$m$^2$/s]', fontsize=_FS_L)
        ax.set_title(f'{visc_label}', fontsize=_FS_T)
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    plt.tight_layout(); stamp_version(fig)
    fig.savefig(str(save_dir / 'D_vs_bead_by_viscosity.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: D_vs_bead_by_viscosity.png')


print('Trend plot functions loaded.')


## Pipeline Utilities (Metadata & Output)

In [ ]:
# ============================================================================
# PIPELINE UTILITIES (metadata, output, version stamp)
# ============================================================================

def stamp_version(fig):
    """Add pipeline version label to bottom-right of figure."""
    fig.text(0.99, 0.005, f'Tracker Pipeline v{PIPELINE_VERSION}',
             fontsize=6, color='gray', alpha=0.5, ha='right', va='bottom',
             transform=fig.transFigure)


def load_or_create_metadata(figures_dir, filepath, file_format, input_units,
                            fps, n_frames, n_particles, bead_diameter_um,
                            solute_pct, solute_type, temp_c):
    """Load or create readme.txt. Returns: (config, should_process)"""
    readme_path = figures_dir / 'readme.txt'
    config = configparser.ConfigParser()
    if readme_path.exists():
        config.read(str(readme_path), encoding='utf-8')
        stored_version = config.get('volatile', 'pipeline_version', fallback='0.0')
        if stored_version == PIPELINE_VERSION and not FORCE_REPROCESS:
            return config, False
        print(f'    [metadata] Version {stored_version} -> {PIPELINE_VERSION}')
        return config, True
    config['constants'] = {
        'filename': Path(filepath).name, 'format': file_format.upper(),
        'input_units': input_units, 'fps': str(fps),
        'total_frames': str(n_frames), 'n_particles': str(n_particles),
        'bead_diameter_um': str(bead_diameter_um),
        'solute_pct': f'{solute_pct:.2f}', 'solute_type': solute_type,
        'temp_c': str(temp_c), 'pixel_size_um': str(PIXEL_SIZE_DEFAULT),
    }
    config['volatile'] = {}
    with open(str(readme_path), 'w', encoding='utf-8') as f:
        f.write(f'# === TRACKER DATA METADATA (readme.txt) ===\n')
        f.write(f'# Generated: {datetime.now().isoformat()}\n')
        f.write(f'# Pipeline: v{PIPELINE_VERSION}\n\n')
        config.write(f)
    return config, True


def update_volatile_metadata(figures_dir, D_var, D_gauss, D_msd,
                             D_var_err, D_gauss_err, D_msd_err,
                             alpha, alpha_err, n_segments,
                             D_mid=0, D_wall=0, eta_corrected=0):
    """Update [volatile] section of readme.txt."""
    readme_path = figures_dir / 'readme.txt'
    if not readme_path.exists(): return
    config = configparser.ConfigParser()
    config.read(str(readme_path), encoding='utf-8')
    config['volatile'] = {
        'pipeline_version': PIPELINE_VERSION,
        'd_variance': f'{D_var:.6f}', 'd_variance_err': f'{D_var_err:.6f}',
        'd_gauss': f'{D_gauss:.6f}', 'd_gauss_err': f'{D_gauss_err:.6f}',
        'd_msd': f'{D_msd:.6f}', 'd_msd_err': f'{D_msd_err:.6f}',
        'd_mid': f'{D_mid:.6f}', 'd_wall': f'{D_wall:.6f}',
        'eta_corrected': f'{eta_corrected:.8f}',
        'alpha': f'{alpha:.4f}', 'alpha_err': f'{alpha_err:.4f}',
        'n_segments': str(n_segments),
        'last_processed': datetime.now().isoformat(),
    }
    with open(str(readme_path), 'w', encoding='utf-8') as f:
        f.write(f'# === TRACKER DATA METADATA (readme.txt) ===\n')
        f.write(f'# Updated: {datetime.now().isoformat()}\n')
        f.write(f'# Pipeline: v{PIPELINE_VERSION}\n\n')
        config.write(f)


def write_trackresults(figures_dir, segments, input_units):
    """Write position data in MTrack2-compatible format."""
    if not segments: return
    out_path = figures_dir / 'trackresults.txt'
    max_len = max(seg['length'] for seg in segments)
    with open(str(out_path), 'w', encoding='utf-8') as f:
        headers = ['Frame']
        for seg in segments:
            headers.extend([f'X{seg["id"]}', f'Y{seg["id"]}', f'Flag{seg["id"]}'])
        f.write('\t'.join(headers) + '\n')
        for frame in range(max_len):
            row = [str(frame + 1)]
            for seg in segments:
                if frame < seg['length']:
                    row.extend([f'{seg["x"][frame]:.4f}', f'{seg["y"][frame]:.4f}', '0'])
                else:
                    row.extend(['', '', ''])
            f.write('\t'.join(row) + '\n')


def write_summary_txt(figures_dir, filepath, file_format, input_units, bead_um,
                      solute_pct, solute_label, temp_c, viscosity, fps, fps_detected,
                      dt, n_frames, n_particles, n_segments, n_steps,
                      disp, msd, theory):
    """Write human-readable analysis summary."""
    T_K = temp_c + 273.15
    lines = []
    lines.append('=' * 70)
    lines.append('ANALYSIS SUMMARY (Pipeline v' + PIPELINE_VERSION + ')')
    lines.append('=' * 70)
    lines.append(f'')
    lines.append(f'Input file: {filepath}')
    lines.append(f'Format: {file_format.upper()} | Units: {input_units}')
    lines.append(f'Bead diameter: {bead_um} um')
    lines.append(f'Solute: {solute_pct:.1f}% {solute_label}')
    lines.append(f'Temperature: {temp_c} C ({T_K:.2f} K)')
    lines.append(f'Viscosity (base): {viscosity*1e3:.3f} mPa.s')
    lines.append(f'Viscosity (Einstein-corrected): {theory["eta_corrected"]*1e3:.3f} mPa.s (phi={theory["phi"]:.2e})')
    lines.append(f'Pixel size: {PIXEL_SIZE_DEFAULT} um/px')
    lines.append(f'Frame rate: {fps} fps {"(from file)" if fps_detected else "(default)"}')
    lines.append(f'dt: {dt*1000:.2f} ms')
    lines.append(f'')
    lines.append(f'Data: {n_frames} frames, {n_particles} particles, {n_segments} segments, {n_steps} steps')
    lines.append(f'')
    lines.append(f'Diffusion Coefficients (um^2/s):')
    lines.append(f'  Direct Variance: {disp["D_var"]:.4f} +/- {disp["D_var_err"]:.4f}')
    lines.append(f'  Gaussian Fit:    {disp["D_gauss"]:.4f} +/- {disp["D_gauss_err"]:.4f}')
    lines.append(f'  MSD Slope:       {msd["D_msd"]:.4f} +/- {msd["D_msd_err"]:.4f}')
    lines.append(f'')
    lines.append(f'Theory (Stokes-Einstein + dual-wall Faxen):')
    lines.append(f'  D_0 (uncorrected):  {theory["D_0"]:.4f} um^2/s')
    lines.append(f'  F_mid (midplane):   {theory["F_mid"]:.4f} (h={CHAMBER_DEPTH_UM/2:.1f} um)')
    lines.append(f'  F_wall (quarter):   {theory["F_wall"]:.4f} (h={CHAMBER_DEPTH_UM/8:.1f} um)')
    lines.append(f'  D_mid:              {theory["D_mid"]:.4f} um^2/s')
    lines.append(f'  D_wall:             {theory["D_wall"]:.4f} um^2/s')
    lines.append(f'')
    lines.append(f'Deviations from D_mid:')
    for name, D_val in [('Variance', disp['D_var']), ('Gaussian', disp['D_gauss']),
                         ('MSD', msd['D_msd'])]:
        dev = (D_val - theory['D_mid']) / theory['D_mid'] * 100 if theory['D_mid'] > 0 else 0
        lines.append(f'  {name}: {dev:+.1f}%')
    lines.append(f'')
    lines.append(f'MSD Exponent: alpha = {msd["alpha"]:.3f} +/- {msd["alpha_err"]:.3f} ({msd["motion_type"]})')
    lines.append(f'MSD fit: forced through origin (MSD = 4*D*tau)')
    lines.append(f'Sigma-clipping: applied before MSD (consistent with variance method)')
    lines.append(f'')
    lines.append(f'Output: {figures_dir}')
    lines.append('=' * 70)
    with open(str(figures_dir / 'summary.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))


print('Pipeline utilities loaded.')


## Batch Processing Loop

Iterates through JOBS: parse -> load -> segment -> analyze -> plot -> save.
Skips datasets whose `readme.txt` already has `pipeline_version = 1.2` (unless `FORCE_REPROCESS = True`).

In [ ]:
# ============================================================================
# BATCH PROCESSING LOOP
# ============================================================================

FIGURES_ROOT.mkdir(parents=True, exist_ok=True)

n_total = len(JOBS)
n_processed = 0
n_skipped = 0
n_failed = 0
batch_results = []

for job_idx, (filepath, temp_c) in enumerate(JOBS, 1):
    filepath = Path(filepath)
    print(f'\n{"="*70}')
    print(f'[{job_idx}/{n_total}] {filepath.name}')

    if not filepath.exists():
        print(f'  FILE NOT FOUND: {filepath}'); n_failed += 1; continue

    try:
        # 1. Parse filename
        bead_um, solute_pct, solute_type, prefix, trial_num, stock_vol, total_vol = parse_filename(filepath)
        if bead_um is None:
            print(f'  WARNING: Could not parse bead size, defaulting to 1.0 um'); bead_um = 1.0
        solute_label = solute_type.capitalize()

        # 2. Output folder
        date_str = filepath.parent.name
        figures_dir = FIGURES_ROOT / date_str / filepath.stem
        figures_dir.mkdir(parents=True, exist_ok=True)

        # 3. Detect format & load
        file_format = detect_format(filepath)
        raw_data, n_frames, n_particles, fps_detected, input_units = load_position_data(filepath, file_format)
        pixel_size = 1.0 if input_units == 'um' else PIXEL_SIZE_DEFAULT
        fps = fps_detected if fps_detected else FPS_DEFAULT
        dt = 1.0 / fps
        pos_unit_label = r'$\mu\mathrm{m}$' if input_units == 'um' else 'pixels'
        px_label = r'$\mu\mathrm{m}$' if input_units == 'um' else 'px'

        print(f'  {file_format.upper()} | {input_units} | {n_frames} frames, {n_particles} particles')
        print(f'  {bead_um} um beads, {solute_pct:.1f}% {solute_label}, {temp_c}C, {fps:.0f} fps')
        print(f'  Slide: {prefix}, Trial: {trial_num}')

        # 4. Metadata check (skip if up-to-date)
        config, should_process = load_or_create_metadata(
            figures_dir, filepath, file_format, input_units,
            fps, n_frames, n_particles, bead_um, solute_pct, solute_type, temp_c)

        if not should_process:
            print(f'  SKIP (v{PIPELINE_VERSION} already processed)')
            n_skipped += 1
            try:
                theory = compute_theory_D(bead_um, solute_pct, solute_type, temp_c, stock_vol, total_vol)
                batch_results.append({
                    'video': filepath.stem, 'date': date_str, 'prefix': prefix, 'trial': trial_num,
                    'bead_um': bead_um, 'solute_pct': solute_pct, 'solute_type': solute_label,
                    'eta_mPas': theory['eta'] * 1e3, 'fps': fps, 'n_segments': int(config.get('volatile','n_segments',fallback='0')),
                    'D_variance': float(config.get('volatile','d_variance',fallback='0')),
                    'D_gauss': float(config.get('volatile','d_gauss',fallback='0')),
                    'D_msd': float(config.get('volatile','d_msd',fallback='0')),
                    'D_var_err': float(config.get('volatile','d_variance_err',fallback='0')),
                    'D_gauss_err': float(config.get('volatile','d_gauss_err',fallback='0')),
                    'D_msd_err': float(config.get('volatile','d_msd_err',fallback='0')),
                    'D_theory': theory['D_0'], 'D_mid': theory['D_mid'], 'D_wall': theory['D_wall'],
                    'F_mid': theory['F_mid'], 'alpha': float(config.get('volatile','alpha',fallback='0')),
                    'alpha_err': float(config.get('volatile','alpha_err',fallback='0')),
                })
            except Exception:
                pass
            continue

        # 5. Segment splitting
        max_jump_data = MAX_JUMP_UM / pixel_size
        segments = split_tracks_at_jumps(raw_data, MIN_SEGMENT_LENGTH, max_jump_data)
        if not segments:
            print(f'  NO VALID SEGMENTS'); n_failed += 1; continue
        n_to_use = len(segments)
        print(f'  Segments: {n_to_use}')

        # 6. Theory (dual-wall Faxen + Einstein viscosity)
        theory = compute_theory_D(bead_um, solute_pct, solute_type, temp_c, stock_vol, total_vol)
        viscosity = theory['eta']
        print(f'  Theory: D_mid={theory["D_mid"]:.4f}, D_wall={theory["D_wall"]:.4f}')

        # 7. Displacement analysis
        disp = compute_displacement_D(segments, pixel_size, dt)
        if disp['n_clipped'] > 0:
            print(f'  Sigma-clip removed {disp["n_clipped"]} outlier steps')

        # 8. MSD analysis (sigma-clips internally, forced through origin)
        msd = compute_msd(segments, pixel_size, dt, fps)

        print(f'  D_var={disp["D_var"]:.4f}  D_gau={disp["D_gauss"]:.4f}  D_msd={msd["D_msd"]:.4f}  '
              f'alpha={msd["alpha"]:.3f} ({msd["motion_type"]})')

        # 9. Write track results
        write_trackresults(figures_dir, segments, input_units)

        # 10. Generate all plots
        plot_trajectories(segments, pixel_size, pos_unit_label, px_label,
                         raw_data, n_particles, n_frames, figures_dir, prefix, trial_num)
        plot_displacement_histograms(disp, dt, figures_dir, prefix, trial_num)
        plot_msd_analysis(msd, dt, figures_dir, prefix, trial_num)
        plot_D_comparison(disp, msd, theory, bead_um, solute_pct, solute_label,
                         fps, figures_dir, prefix, trial_num)
        plot_combined_summary(segments, raw_data, n_particles, n_frames,
                             disp, msd, theory, pixel_size, px_label, pos_unit_label,
                             bead_um, solute_label, solute_pct, temp_c, fps, file_format,
                             figures_dir, prefix, trial_num)

        # 11. Write summary
        write_summary_txt(figures_dir, filepath, file_format, input_units, bead_um,
                         solute_pct, solute_label, temp_c, viscosity, fps, fps_detected,
                         dt, n_frames, n_particles, n_to_use, disp['n_steps'], disp, msd, theory)

        # 12. Update metadata
        update_volatile_metadata(figures_dir, disp['D_var'], disp['D_gauss'], msd['D_msd'],
                                disp['D_var_err'], disp['D_gauss_err'], msd['D_msd_err'],
                                msd['alpha'], msd['alpha_err'], n_to_use,
                                theory['D_mid'], theory['D_wall'], theory['eta_corrected'])

        # 13. Collect for batch
        D_avg = np.mean([disp['D_var'], disp['D_gauss'], msd['D_msd']])
        dev_pct = (D_avg - theory['D_mid']) / theory['D_mid'] * 100 if theory['D_mid'] > 0 else 0
        batch_results.append({
            'video': filepath.stem, 'date': date_str, 'prefix': prefix, 'trial': trial_num,
            'bead_um': bead_um, 'solute_pct': solute_pct, 'solute_type': solute_label,
            'eta_mPas': viscosity * 1e3, 'fps': fps, 'n_segments': n_to_use,
            'D_variance': disp['D_var'], 'D_gauss': disp['D_gauss'], 'D_msd': msd['D_msd'],
            'D_var_err': disp['D_var_err'], 'D_gauss_err': disp['D_gauss_err'], 'D_msd_err': msd['D_msd_err'],
            'D_theory': theory['D_0'], 'D_mid': theory['D_mid'], 'D_wall': theory['D_wall'],
            'F_mid': theory['F_mid'], 'alpha': msd['alpha'], 'alpha_err': msd['alpha_err'],
            'dev_pct': dev_pct,
        })
        n_processed += 1
        print(f'  DONE -> {figures_dir}')

    except Exception as e:
        import traceback
        print(f'  FAILED: {e}'); traceback.print_exc(); n_failed += 1

print(f'\n{"="*70}')
print(f'BATCH COMPLETE: {n_processed} processed, {n_skipped} skipped, {n_failed} failed (of {n_total})')
print(f'{"="*70}')


## Results Summary & Export

In [ ]:
# ============================================================================
# RESULTS SUMMARY & CSV EXPORT
# ============================================================================

if batch_results:
    print(f'\n{"="*140}')
    print(f'{"BATCH RESULTS -- Pipeline v" + PIPELINE_VERSION:^140}')
    print(f'{"="*140}')
    print(f'{"Video":<45} {"Slide":>5} {"T#":>2} {"Bead":>4} {"Solute":>10} '
          f'{"D_var":>8} {"D_gau":>8} {"D_msd":>8} {"D_mid":>8} {"D_wall":>8} {"alpha":>6}')
    print(f'{"-"*140}')
    for r in batch_results:
        sol = f"{r['solute_pct']:.0f}%{r['solute_type'][:3]}" if r['solute_pct'] > 0 else "Water"
        name = r['video'][:43]
        print(f'{name:<45} {r.get("prefix",""):>5} {r.get("trial",1):>2} {r["bead_um"]:>4.1f} {sol:>10} '
              f'{r["D_variance"]:>8.4f} {r["D_gauss"]:>8.4f} {r["D_msd"]:>8.4f} '
              f'{r["D_mid"]:>8.4f} {r["D_wall"]:>8.4f} {r["alpha"]:>6.2f}')
    print(f'{"="*140}')

    csv_path = FIGURES_ROOT / 'batch_results.csv'
    df = pd.DataFrame(batch_results)
    col_order = ['video','date','prefix','trial','bead_um','solute_pct','solute_type',
                 'eta_mPas','fps','n_segments',
                 'D_variance','D_var_err','D_gauss','D_gauss_err','D_msd','D_msd_err',
                 'D_theory','D_mid','D_wall','F_mid','alpha','alpha_err','dev_pct']
    df = df[[c for c in col_order if c in df.columns]]
    df.to_csv(str(csv_path), index=False, float_format='%.6f')
    print(f'\nSaved: {csv_path}')
else:
    print('No results to summarize.')


## Trend Plots (Per-Session & Lab-Level)

Scans ALL `summary.txt` files (including skipped jobs) so trend plots always reflect complete data.

In [ ]:
# ============================================================================
# TREND PLOTS (scan all summary.txt files)
# ============================================================================

all_results = []
for summary_path in sorted(FIGURES_ROOT.rglob('summary.txt')):
    folder = summary_path.parent
    try:
        txt = summary_path.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        txt = summary_path.read_text(encoding='latin-1')

    def _grab(pattern, default=None):
        m = re.search(pattern, txt)
        return m.group(1) if m else default

    video_name = folder.name
    date_str_t = folder.parent.name
    bead_str = _grab(r'Bead diameter:\s*([\d.]+)')
    solute_str = _grab(r'Solute:\s*([\d.]+)%')
    solute_type_s = 'acetone' if 'Acetone' in txt else 'glycerol'
    eta_str = _grab(r'Viscosity \(base\):\s*([\d.]+)\s*mPa')
    if not eta_str:
        eta_str = _grab(r'Viscosity:\s*([\d.]+)\s*mPa')
    fps_str = _grab(r'Frame rate:\s*([\d.]+)')
    d_var_str = _grab(r'Direct Variance:\s*([\d.eE+-]+)')
    d_gau_str = _grab(r'Gaussian Fit:\s*([\d.eE+-]+)')
    d_msd_str = _grab(r'MSD Slope:\s*([\d.eE+-]+)')
    d_mid_str = _grab(r'D_mid:\s*([\d.eE+-]+)')
    d_wall_str = _grab(r'D_wall:\s*([\d.eE+-]+)')
    d_the_str = d_mid_str  # use D_mid as the primary theory value
    if not d_the_str:
        d_the_str = _grab(r'D_theory \(final\):\s*([\d.eE+-]+)')
    alpha_str = _grab(r'alpha\s*=\s*([\d.eE+-]+)')
    alpha_err_str = _grab(r'alpha\s*=\s*[\d.eE+-]+\s*\+/-\s*([\d.eE+-]+)')

    if not all([bead_str, d_var_str, d_msd_str, alpha_str]):
        continue

    bead = float(bead_str)
    eta = float(eta_str) if eta_str else 1.0
    d_var = float(d_var_str)
    d_gau = float(d_gau_str) if d_gau_str else 0.0
    d_msd = float(d_msd_str)
    d_the = float(d_the_str) if d_the_str else 0.0
    d_wall = float(d_wall_str) if d_wall_str else d_the * 0.8
    alpha_v = float(alpha_str)
    alpha_err_v = float(alpha_err_str) if alpha_err_str else 0.0
    solute_pct_v = float(solute_str) if solute_str else 0.0

    # Parse prefix and trial from folder name
    parsed = parse_filename(folder.name + '-tracker.txt')
    pfx = parsed[3] if parsed[3] else video_name[:3]
    tr = parsed[4] if parsed[4] else 1

    if solute_type_s == 'acetone':
        sol_label = f'{solute_pct_v:.0f}%Ace'
    elif solute_pct_v > 0:
        sol_label = f'{solute_pct_v:.0f}%Gly'
    else:
        sol_label = 'Water'

    all_results.append({
        'video': video_name, 'date': date_str_t, 'bead': bead,
        'solute_pct': solute_pct_v, 'solute_type': solute_type_s,
        'solute_label': sol_label, 'eta': eta,
        'd_var': d_var, 'd_gau': d_gau, 'd_msd': d_msd,
        'd_theory': d_the, 'd_wall': d_wall,
        'alpha': alpha_v, 'alpha_err': alpha_err_v,
        'prefix': pfx, 'trial': tr,
    })

print(f'Loaded {len(all_results)} datasets for trend plots.')

if len(all_results) >= 2:
    plot_overall_trends(all_results, FIGURES_ROOT)

    dates = sorted(set(r['date'] for r in all_results))
    for date in dates:
        dr = [r for r in all_results if r['date'] == date]
        plot_date_trends(dr, date, FIGURES_ROOT)

    plot_lab_level_trends(all_results, FIGURES_ROOT)

print(f'\nPipeline v{PIPELINE_VERSION} complete.')
